In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import json
import os
import sys
sys.path.append("/content/drive/MyDrive/MockProject_NguyenTuanPhat/Day 12")
import importlib
importlib.invalidate_caches()

from prompt_template import build_prompt

In [ ]:
INPUT_FILE = "/content/drive/MyDrive/MockProject_NguyenTuanPhat/Day 12/medquad.json"
OUTPUT_FILE = "/content/drive/MyDrive/MockProject_NguyenTuanPhat/Day 12/output/train.jsonl"

In [ ]:
def load_dataset(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

In [ ]:
def validate(sample):
    return (
        isinstance(sample, dict)
        and sample.get("question")
        and sample.get("answer")
    )

In [ ]:
def convert(sample):
    question = sample["question"].strip()
    answer = sample["answer"].strip()

    # Lúc chạy thật, vector DB sẽ trả về các đoạn tài liệu (chunks) liên quan
    # nhất tới câu hỏi. Ở đây ta chưa có vector DB thật, nên dùng chính
    # "answer" gốc trong MedQuAD làm chunk giả lập -- vì answer chính là
    # đoạn tài liệu chứa câu trả lời đúng, tương tự những gì vector DB
    # sẽ tìm ra trong thực tế.
    messages = build_prompt(chunks=[answer], question=question)

    # build_prompt() chỉ trả về [system, user] (chưa có câu trả lời).
    # Thêm message "assistant" để hoàn chỉnh 1 sample train.
    messages.append({"role": "assistant", "content": answer})

    return {"messages": messages}

In [ ]:
def save_jsonl(data, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False))
            f.write("\n")

In [ ]:
def main():

    dataset = load_dataset(INPUT_FILE)

    output = []

    skipped = 0

    for sample in dataset[:300]:

        if not validate(sample):
            skipped += 1
            continue

        output.append(convert(sample))

    save_jsonl(output, OUTPUT_FILE)

    print("=" * 40)
    print(f"Total samples : {len(dataset)}")
    print(f"Converted     : {len(output)}")
    print(f"Skipped       : {skipped}")
    print(f"Output        : {OUTPUT_FILE}")

In [ ]:
if __name__ == "__main__":
    main()

Total samples : 11548
Converted     : 300
Skipped       : 0
Output        : /content/drive/MyDrive/MockProject_NguyenTuanPhat/Day 12/output/train.jsonl
